# Paper 2 · Langkah 5: elemen desain di dalam dan di luar area sorotan (RQ2 dan RQ3)

Notebook ini mengukur apa yang ada di area yang disorot model, memakai ukuran elemen desain.

Untuk setiap gambar:
1. Area sorotan **R** = 20% piksel teratas menurut peta (diulang pada 10% dan 30%).
2. Sebelas ukuran dihitung **di dalam R** dan **di luar R**: hue, saturasi, kekayaan warna, kecerahan, kontras, gradasi, kepadatan garis tepi, dan empat ukuran tekstur.
3. **Kontrol acak-tukar**: area R milik gambar lain (dari sumber yang sama) ditempelkan ke gambar ini. Kalau selisih dalam-luar yang asli tidak lebih besar daripada versi tukar, berarti selisih itu hanya soal posisi, bukan soal isi gambar.
4. Ukuran komposisi R: jarak dari tengah, sebaran, dan jumlah gumpalan.

Peta utama: **occlusion**. Pembanding: **Grad-CAM**. Tidak butuh GPU.

Hasil tiap bagian langsung disimpan ke Drive. Kalau sesi terputus, klik Run all lagi; bagian yang sudah selesai dilewati.

## 1. Persiapan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, io, json, time, shutil, zipfile, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn.functional as F
import cv2, scipy
from scipy import stats, ndimage
from PIL import Image
import PIL

# Pengaturan (jangan diubah tanpa mencatatnya di log kerja)
SEED = 42
INPUT = 224
PATCH, STRIDE = 32, 16
THRESHOLDS = [0.20, 0.10, 0.30]      # 0.20 = utama; 0.10 dan 0.30 = uji kepekaan
HUE_BINS, GLCM_LEVELS = 36, 32
CANNY_LOW, CANNY_HIGH = 100, 200
CHROMA_MIN = 0.10                    # piksel dengan saturasi/kecerahan di bawah ini tidak dihitung untuk hue

PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
MAPS = RESULTS / 'peta'
ELEM = RESULTS / 'elemen'
CACHE = PROJECT / 'data_olahan'
LOCAL = Path('/content/work')
for p in [ELEM, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\nnumpy: {np.__version__}\n'
       f'scipy: {scipy.__version__}\nopencv: {cv2.__version__}\npillow: {PIL.__version__}\ntorch: {torch.__version__}\n'
       f'ambang area sorotan: {THRESHOLDS}; hue bins: {HUE_BINS}; GLCM level: {GLCM_LEVELS}; Canny: {CANNY_LOW}/{CANNY_HIGH}\n')
(NOTES / 'lingkungan_langkah5.txt').write_text(env)
print(env)

## 2. Data: gambar yang dilihat model

In [ ]:
manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
assert len(manifest) == 3000 and 'fold' in manifest.columns, 'Jalankan notebook langkah 2 dulu.'
manifest['y'] = (manifest['label'] == 'ai').astype(int)
assert len(list(MAPS.glob('*.npz'))) == 12, 'Peta belum lengkap. Jalankan notebook langkah 4 sampai 12 dari 12.'

local_raw = LOCAL / 'raw'
raw_zip = CACHE / 'ai_artbench_sample_raw.zip'
t0 = time.time()
if not local_raw.exists():
    shutil.copy(raw_zip, LOCAL / 'raw.zip')
    with zipfile.ZipFile(LOCAL / 'raw.zip') as z:
        z.extractall(local_raw)
manifest['raw_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)

def to_norm(im):
    im = im.convert('RGB')
    w, h = im.size
    if w != h:
        s = min(w, h); left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    buf = io.BytesIO()
    im.save(buf, 'JPEG', quality=90, subsampling=0, optimize=False)
    return buf.getvalue()

def make_condition(row, cond):
    with Image.open(row.raw_path) as im:
        im.load()
        if cond == 'riwayat_disamakan' and row.source == 'human':
            buf = io.BytesIO()
            im.convert('RGB').save(buf, 'JPEG', quality=75, subsampling=2, optimize=False)
            im = Image.open(io.BytesIO(buf.getvalue())); im.load()
        return Image.open(io.BytesIO(to_norm(im))).convert('RGB')

def load_condition(cond):
    def _one(row):
        im = make_condition(row, cond)
        if im.size != (INPUT, INPUT):
            im = im.resize((INPUT, INPUT), Image.BICUBIC)
        return np.asarray(im, dtype=np.uint8)
    with ThreadPoolExecutor(8) as ex:
        return np.stack(list(ex.map(_one, manifest.itertuples())))      # N,H,W,3 uint8

DATA = {c: load_condition(c) for c in ['riwayat_disamakan', 'normal']}
idx_of = {s: i for i, s in enumerate(manifest['sample_id'])}
src_all = manifest['source'].values
print(f'Data siap ({time.time() - t0:.0f} detik):', {k: v.shape for k, v in DATA.items()})

## 3. Fungsi ukuran

Sebelas ukuran per area, dikelompokkan menurut elemen desain:
- **Hue**: entropi hue (seberapa beragam rona warnanya).
- **Warna**: saturasi rata-rata dan kekayaan warna (Hasler dan Süsstrunk).
- **Nada**: kecerahan, kontras (simpangan baku luminans), gradasi (rata-rata besar gradien Sobel).
- **Garis**: kepadatan garis tepi (Canny).
- **Tekstur**: empat ukuran GLCM, yaitu kontras, homogenitas, energi, korelasi.

Komposisi diukur pada bentuk area R itu sendiri.

In [ ]:
NPIX = INPUT * INPUT
MEASURES = ['hue_entropy', 'saturation', 'colourfulness', 'brightness', 'contrast', 'gradient',
            'edge_density', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation']
LABEL = {'hue_entropy': 'Hue: entropi', 'saturation': 'Warna: saturasi', 'colourfulness': 'Warna: kekayaan',
         'brightness': 'Nada: kecerahan', 'contrast': 'Nada: kontras', 'gradient': 'Nada: gradasi',
         'edge_density': 'Garis: kepadatan tepi', 'glcm_contrast': 'Tekstur: kontras',
         'glcm_homogeneity': 'Tekstur: homogenitas', 'glcm_energy': 'Tekstur: energi', 'glcm_correlation': 'Tekstur: korelasi'}

# Pasangan piksel bertetangga untuk GLCM (4 arah, jarak 1)
_flat = np.arange(NPIX).reshape(INPUT, INPUT)
PAIRS = []
for dy, dx in [(0, 1), (1, 0), (1, 1), (1, -1)]:
    a = _flat[max(0, -dy):INPUT - max(0, dy), max(0, -dx):INPUT - max(0, dx)]
    b = _flat[max(0, dy):INPUT - max(0, -dy), max(0, dx):INPUT - max(0, -dx)]
    PAIRS.append((a.ravel(), b.ravel()))
_lv = np.arange(GLCM_LEVELS, dtype=np.float64)
_I, _J = np.meshgrid(_lv, _lv, indexing='ij')
_D2 = (_I - _J) ** 2

def precompute(img):
    f = {}
    rgb = img.astype(np.float32)
    hsv = cv2.cvtColor(rgb / 255.0, cv2.COLOR_RGB2HSV)            # H 0-360, S 0-1, V 0-1
    f['hbin'] = np.minimum((hsv[..., 0] / 360.0 * HUE_BINS).astype(np.int64), HUE_BINS - 1).ravel()
    f['sat'] = hsv[..., 1].ravel(); f['val'] = hsv[..., 2].ravel()
    f['chroma_ok'] = (f['sat'] > CHROMA_MIN) & (f['val'] > CHROMA_MIN)
    f['rg'] = (rgb[..., 0] - rgb[..., 1]).ravel()
    f['yb'] = (0.5 * (rgb[..., 0] + rgb[..., 1]) - rgb[..., 2]).ravel()
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f['lum'] = gray.astype(np.float32).ravel()
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    f['grad'] = np.sqrt(gx ** 2 + gy ** 2).ravel()
    f['edge'] = (cv2.Canny(gray, CANNY_LOW, CANNY_HIGH) > 0).ravel()
    q = (gray.astype(np.int64) * GLCM_LEVELS // 256).ravel()
    f['codes'] = [q[a] * GLCM_LEVELS + q[b] for a, b in PAIRS]
    return f

def region_measures(f, m):
    out = dict.fromkeys(MEASURES, np.nan)
    if m.sum() < 100:
        return out
    sel = m & f['chroma_ok']
    if sel.sum() >= 50:
        h = np.bincount(f['hbin'][sel], minlength=HUE_BINS).astype(np.float64); p = h / h.sum(); p = p[p > 0]
        out['hue_entropy'] = float(-(p * np.log2(p)).sum())
    out['saturation'] = float(f['sat'][m].mean())
    rg, yb = f['rg'][m], f['yb'][m]
    out['colourfulness'] = float(np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2))
    out['brightness'] = float(f['val'][m].mean())
    out['contrast'] = float(f['lum'][m].std())
    out['gradient'] = float(f['grad'][m].mean())
    out['edge_density'] = float(f['edge'][m].mean())
    P = np.zeros(GLCM_LEVELS * GLCM_LEVELS)
    for (a, b), code in zip(PAIRS, f['codes']):
        ok = m[a] & m[b]
        P += np.bincount(code[ok], minlength=GLCM_LEVELS * GLCM_LEVELS)
    if P.sum() > 0:
        P = P.reshape(GLCM_LEVELS, GLCM_LEVELS); P = P + P.T; P /= P.sum()
        out['glcm_contrast'] = float((P * _D2).sum())
        out['glcm_homogeneity'] = float((P / (1 + _D2)).sum())
        out['glcm_energy'] = float(np.sqrt((P ** 2).sum()))
        mu = (P.sum(1) * _lv).sum(); var = (P.sum(1) * (_lv - mu) ** 2).sum()
        out['glcm_correlation'] = float((P * (_I - mu) * (_J - mu)).sum() / var) if var > 1e-12 else np.nan
    return out

GRID = max(1, INPUT // 32)
def composition(m):
    m2 = m.reshape(INPUT, INPUT); ys, xs = np.nonzero(m2); c = (INPUT - 1) / 2
    cell = m2[:GRID * (INPUT // GRID), :GRID * (INPUT // GRID)].reshape(GRID, INPUT // GRID, GRID, INPUT // GRID).sum((1, 3)).ravel().astype(np.float64)
    p = cell / cell.sum(); p = p[p > 0]
    lab, ncomp = ndimage.label(m2, structure=np.ones((3, 3)))
    sizes = np.bincount(lab.ravel())[1:]
    return {'comp_centroid_dist': float(np.sqrt((ys.mean() - c) ** 2 + (xs.mean() - c) ** 2) / (c * 2 ** 0.5)),
            'comp_spatial_entropy': float(-(p * np.log2(p)).sum() / np.log2(GRID * GRID)) if GRID > 1 else np.nan,
            'comp_n_components': int(ncomp), 'comp_largest_share': float(sizes.max() / sizes.sum())}

def patch_to_cells(d):
    k, n, C = PATCH // STRIDE, d.shape[-1], INPUT // STRIDE
    acc = torch.zeros(*d.shape[:-2], C, C); cnt = torch.zeros(C, C)
    for a in range(k):
        for b in range(k):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def rank_maps(maps, rng):
    n = len(maps); rank = np.empty((n, NPIX), dtype=np.uint16)
    for s in range(0, n, 200):
        flat = maps[s:s + 200].reshape(-1, NPIX).astype(np.float64)
        flat = flat + 1e-9 * rng.random(flat.shape)                 # pemecah nilai kembar
        order = np.argsort(-flat, axis=1)
        np.put_along_axis(rank[s:s + 200], order, np.arange(NPIX, dtype=np.uint16)[None].repeat(len(order), 0), axis=1)
    return rank
ALL = np.ones(NPIX, dtype=bool)
print('Fungsi siap.', len(MEASURES), 'ukuran per area.')

## 4. Hitung ukuran untuk semua bagian

Dua belas bagian, sama seperti langkah 4. Untuk kondisi **riwayat_disamakan** dan **lintas generator** dipakai tiga ambang (10%, 20%, 30%); untuk kondisi **normal** hanya 20%.

Satu baris muncul tiap satu bagian selesai.

In [ ]:
def process(name, cond, thresholds):
    out_file = ELEM / f'{name}.csv'
    if out_file.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    t0 = time.time()
    z = np.load(MAPS / f'{name}.npz', allow_pickle=True)
    ids = z['sample_id'].astype(str); idx = np.array([idx_of[s] for s in ids]); pred = z['pred'].astype(int)
    n = len(idx); ar = np.arange(n)
    rng = np.random.default_rng(SEED + sum(map(ord, name)))
    gc = torch.from_numpy(z['gc'][ar, pred]).float()
    occ = torch.from_numpy(z['occ'][ar, pred]).float()
    cells = patch_to_cells(occ).clamp(min=0)
    empty = {'occlusion': (cells.amax((1, 2)) <= 0).numpy(), 'gradcam': (gc.amax((1, 2)) <= 0).numpy()}
    ranks = {'occlusion': rank_maps(up(cells).numpy(), rng), 'gradcam': rank_maps(up(gc).numpy(), rng)}
    occ_max = occ.amax((1, 2)).numpy(); p_pred = z['p'][ar, pred]
    # Pasangan acak-tukar: gambar lain dari sumber yang sama di bagian ini
    partner = ar.copy(); s_here = src_all[idx]
    for s in np.unique(s_here):
        loc = rng.permutation(np.where(s_here == s)[0])
        if len(loc) > 1:
            partner[loc] = np.roll(loc, -1)
    X = DATA[cond]; rows = []
    for k in range(n):
        f = precompute(X[idx[k]])
        whole = {f'whole_{m}': v for m, v in region_measures(f, ALL).items()}
        for method in ('occlusion', 'gradcam'):
            for thr in thresholds:
                row = {'bagian': name, 'sample_id': ids[k], 'metode': method, 'ambang': thr, 'pred': int(pred[k]),
                       'p_pred': float(p_pred[k]), 'occ_max_drop': float(occ_max[k]), 'peta_kosong': bool(empty[method][k])}
                row.update(whole)
                if not empty[method][k]:
                    m = ranks[method][k] < thr * NPIX
                    for tag, mask in (('in', m), ('out', ~m)):
                        row.update({f'{tag}_{a}': v for a, v in region_measures(f, mask).items()})
                    row.update(composition(m))
                    if partner[k] != k and not empty[method][partner[k]]:
                        ms = ranks[method][partner[k]] < thr * NPIX
                        for tag, mask in (('shin', ms), ('shout', ~ms)):
                            row.update({f'{tag}_{a}': v for a, v in region_measures(f, mask).items()})
                rows.append(row)
    pd.DataFrame(rows).to_csv(out_file, index=False)
    print(f'{name}: {n} gambar, {time.time() - t0:.0f} detik')

UNITS = ([(f'riwayat_disamakan_fold{k}', 'riwayat_disamakan', THRESHOLDS) for k in range(5)] +
         [('lintas2_riwayat_disamakan_latihLD_ujiSD', 'riwayat_disamakan', THRESHOLDS),
          ('lintas2_riwayat_disamakan_latihSD_ujiLD', 'riwayat_disamakan', THRESHOLDS)] +
         [(f'normal_fold{k}', 'normal', [0.20]) for k in range(5)])
for name, cond, thr in UNITS:
    process(name, cond, thr)

## 5. Statistik dan ringkasan

Untuk setiap ukuran dihitung **selisih dalam dikurangi luar**, lalu dibandingkan dengan selisih yang sama pada kontrol acak-tukar (uji Wilcoxon berpasangan). Ukuran efek **r** berkisar dari -1 sampai 1:
- **r positif**: area sorotan punya ukuran itu lebih tinggi daripada yang bisa dijelaskan posisi saja.
- **r negatif**: lebih rendah.
- Tanda **\*** berarti bermakna setelah koreksi uji ganda (FDR 5%).

In [ ]:
df = pd.concat([pd.read_csv(f) for f in sorted(ELEM.glob('*.csv'))], ignore_index=True)
df = df.join(manifest.set_index('sample_id')[['label', 'source', 'style', 'fold', 'y']], on='sample_id')
df['hasil'] = np.select([(df.y == 1) & (df.pred == 1), (df.y == 0) & (df.pred == 0), (df.y == 0) & (df.pred == 1)],
                        ['AI ditebak AI', 'manusia ditebak manusia', 'manusia ditebak AI'], 'AI ditebak manusia')
df['keluarga'] = np.where(df.bagian.str.startswith('lintas2'), 'lintas', np.where(df.bagian.str.startswith('normal'), 'normal', 'lima_fold'))
print('Bagian selesai:', df.bagian.nunique(), 'dari 12 | baris:', len(df))

SRC = {'human': 'manusia', 'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
def group_label(r):
    s = SRC[r['source']]
    if r['keluarga'] == 'lintas':
        return f'lintas: {s} lolos' if r['hasil'] == 'AI ditebak manusia' else (f'lintas: {s} benar' if r['hasil'] in ('manusia ditebak manusia', 'AI ditebak AI') else f'lintas: {s} dicap AI')
    pre = '5-fold' if r['keluarga'] == 'lima_fold' else 'normal'
    return f'{pre}: {s} benar' if r['hasil'] in ('manusia ditebak manusia', 'AI ditebak AI') else f'{pre}: {s} salah'
df['kelompok'] = df.apply(group_label, axis=1)
df.to_csv(RESULTS / 'rq2_elemen_per_gambar.csv', index=False)

def paired_effect(x):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; x = x[x != 0]
    if len(x) < 10:
        return np.nan, np.nan, len(x)
    r = stats.rankdata(np.abs(x)); pos, neg = r[x > 0].sum(), r[x < 0].sum()
    return (pos - neg) / (pos + neg), stats.wilcoxon(x).pvalue, len(x)

def bh(p):
    p = np.asarray(p, dtype=float); out = np.full(len(p), np.nan); ok = ~np.isnan(p)
    if ok.sum() == 0: return out
    pv = p[ok]; order = np.argsort(pv); ranked = pv[order] * len(pv) / (np.arange(len(pv)) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]; res = np.empty(len(pv)); res[order] = np.minimum(ranked, 1)
    out[ok] = res; return out

rows = []
for (fam, method, thr, grp), g in df[~df.peta_kosong].groupby(['keluarga', 'metode', 'ambang', 'kelompok']):
    for m in MEASURES:
        d = g[f'in_{m}'] - g[f'out_{m}']; ds = g[f'shin_{m}'] - g[f'shout_{m}']
        r, p, n = paired_effect(d - ds)
        rows.append({'keluarga': fam, 'metode': method, 'ambang': thr, 'kelompok': grp, 'ukuran': m, 'n': n,
                     'median_dalam': g[f'in_{m}'].median(), 'median_luar': g[f'out_{m}'].median(),
                     'median_selisih': d.median(), 'median_selisih_tukar': ds.median(), 'r': r, 'p': p})
S = pd.DataFrame(rows)
S['uji'] = np.where(S.keluarga == 'normal', 'normal', 'utama')
S['p_fdr'] = S.groupby(['uji', 'metode', 'ambang'])['p'].transform(bh)
S['bermakna'] = S.p_fdr < 0.05
S.to_csv(RESULTS / 'rq2_elemen_ringkasan.csv', index=False)

def show(method, thr, groups, title):
    t = S[(S.metode == method) & (np.isclose(S.ambang, thr)) & (S.kelompok.isin(groups))].copy()
    t['sel'] = t.r.map(lambda v: 'nan' if pd.isna(v) else f'{v:+.2f}') + np.where(t.bermakna, '*', ' ')
    pv = t.pivot(index='ukuran', columns='kelompok', values='sel').reindex(MEASURES)[[g for g in groups if g in t.kelompok.unique()]]
    pv.index = [LABEL[i] for i in pv.index]
    print('\n' + title); print(pv.to_string())

MAIN = ['5-fold: manusia benar', '5-fold: LD benar', '5-fold: SD benar', 'lintas: LD lolos', 'lintas: SD lolos', 'lintas: manusia benar']
NORM = ['normal: manusia benar', 'normal: LD benar', 'normal: SD benar']
print('\nJumlah gambar per kelompok (occlusion, ambang 20%):')
print(df[(df.metode == 'occlusion') & np.isclose(df.ambang, 0.2)].groupby('kelompok').agg(n=('sample_id', 'size'), peta_kosong=('peta_kosong', 'sum')).to_string())

show('occlusion', 0.20, MAIN, 'A. UTAMA: occlusion, area 20% (r; * = bermakna)')
show('gradcam', 0.20, MAIN, 'B. PEMBANDING: Grad-CAM, area 20%')
show('occlusion', 0.10, MAIN, 'C. KEPEKAAN: occlusion, area 10%')
show('occlusion', 0.30, MAIN, 'D. KEPEKAAN: occlusion, area 30%')
show('occlusion', 0.20, NORM, 'E. KONDISI NORMAL: occlusion, area 20%')

t = S[(S.metode == 'occlusion') & np.isclose(S.ambang, 0.2) & S.kelompok.isin(MAIN)]
print('\nF. BESARAN (occlusion, 20%): median dalam | median luar | selisih asli | selisih tukar')
for grp in MAIN:
    g = t[t.kelompok == grp].set_index('ukuran').reindex(MEASURES)
    if g['n'].notna().any():
        g.index = [LABEL[i] for i in g.index]
        print('\n' + grp); print(g[['median_dalam', 'median_luar', 'median_selisih', 'median_selisih_tukar']].round(3).to_string())

c = df[(df.metode == 'occlusion') & np.isclose(df.ambang, 0.2) & ~df.peta_kosong & df.kelompok.isin(MAIN + NORM)]
print('\nG. KOMPOSISI area sorotan (occlusion, 20%; median)')
print(c.groupby('kelompok')[['comp_centroid_dist', 'comp_spatial_entropy', 'comp_n_components', 'comp_largest_share', 'occ_max_drop']].median().round(3).reindex([g for g in MAIN + NORM if g in c.kelompok.unique()]).to_string())

# Ukuran seluruh gambar per sumber (konteks): apakah gambar AI memang berbeda secara keseluruhan?
w = df[(df.keluarga == 'lima_fold') & (df.metode == 'occlusion') & np.isclose(df.ambang, 0.2)]
wr = []
for m in MEASURES:
    row = {'ukuran': LABEL[m]}
    hv = w[w.source == 'human'][f'whole_{m}'].dropna()
    row['median manusia'] = hv.median()
    for s in ('latent_diffusion', 'stable_diffusion'):
        av = w[w.source == s][f'whole_{m}'].dropna()
        u, p = stats.mannwhitneyu(av, hv, alternative='two-sided')
        row[f'median {SRC[s]}'] = av.median(); row[f'delta {SRC[s]} vs manusia'] = 2 * u / (len(av) * len(hv)) - 1; row[f'p {SRC[s]}'] = p
    wr.append(row)
W = pd.DataFrame(wr).set_index('ukuran'); W.to_csv(RESULTS / 'rq2_elemen_seluruh_gambar.csv')
print('\nH. SELURUH GAMBAR per sumber (delta: -1 sampai 1; positif = AI lebih tinggi daripada manusia)')
print(W.round(3).to_string())

# RQ3: perbedaan antar gaya (occlusion, 20%, kelompok 5-fold yang ditebak benar)
st = df[(df.keluarga == 'lima_fold') & (df.metode == 'occlusion') & np.isclose(df.ambang, 0.2) & ~df.peta_kosong & df.kelompok.str.endswith('benar')].copy()
kw, per_style = [], []
for m in MEASURES:
    st['_x'] = (st[f'in_{m}'] - st[f'out_{m}']) - (st[f'shin_{m}'] - st[f'shout_{m}'])
    for s, g in st.groupby('source'):
        parts = [v['_x'].dropna().values for _, v in g.groupby('style')]
        h, p = stats.kruskal(*parts) if all(len(v) > 4 for v in parts) else (np.nan, np.nan)
        kw.append({'ukuran': LABEL[m], 'sumber': SRC[s], 'H': h, 'p': p})
        for sty, v in g.groupby('style'):
            r, pp, n = paired_effect(v['_x'].values)
            per_style.append({'ukuran': m, 'sumber': SRC[s], 'gaya': sty, 'n': n, 'r': r, 'p': pp, 'median': v['_x'].median()})
KW = pd.DataFrame(kw); KW['p_fdr'] = bh(KW.p.values)
KW['sel'] = KW.H.map(lambda v: 'nan' if pd.isna(v) else f'{v:.1f}') + np.where(KW.p_fdr < 0.05, '*', ' ')
PS = pd.DataFrame(per_style); PS.to_csv(RESULTS / 'rq3_elemen_per_gaya.csv', index=False); KW.to_csv(RESULTS / 'rq3_uji_antar_gaya.csv', index=False)
print('\nI. BEDA ANTAR GAYA (Kruskal-Wallis H; * = bermakna setelah FDR)')
print(KW.pivot(index='ukuran', columns='sumber', values='sel').reindex([LABEL[m] for m in MEASURES]).to_string())
for s in ('manusia', 'LD', 'SD'):
    print(f'\nJ. r per gaya, sumber {s} (occlusion, 20%)')
    pv = PS[PS.sumber == s].pivot(index='ukuran', columns='gaya', values='r').reindex(MEASURES)
    pv.index = [LABEL[i] for i in pv.index]; pv.columns = [c[:6] for c in pv.columns]
    print(pv.round(2).to_string())
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tertulis "Bagian selesai: 12 dari 12" dan tabel A sampai J muncul, semua beres. Tekan **Cmd+S**, lalu kabari Claude.